In [5]:
# =============================================================================
# TASK 1: Graph Concepts & State Design
# =============================================================================
# LANGGRAPH CORE BUILDING BLOCKS:
#
# 1. StateGraph — Container for your workflow (like a blank flowchart)
# 2. State — Shared TypedDict that all nodes read from and write to
# 3. Node — A function that takes state, does work, returns updated fields
# 4. Edge — Transition to next node (add_edge "plan" -> "execute")
# 5. Conditional Edge — Smart transition: "if score >= 7 go here, else go there"
# 6. Entry Point — Where execution starts (set_entry_point)
# 7. END — Special node that stops the graph
# =============================================================================

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
import operator

# --- STATE SCHEMA ---
class ResearchState(TypedDict):
    query: str
    plan: str
    search_results: str
    answer: str
    quality_score: int
    retry_count: int
    max_retries: int

# --- ASCII DIAGRAM OF THE GRAPH ---
diagram = """
+-------------------------------------------------------------------+
|                    RESEARCH ASSISTANT GRAPH                        |
+-------------------------------------------------------------------+
|  +----------+                                                     |
|  |  START   |                                                     |
|  +----+-----+                                                     |
|       v                                                            |
|  +----------+    "Create a plan based on the query"               |
|  |   PLAN   +----------------------------------------+            |
|  +----+-----+                                        |            |
|       v                                              |            |
|  +----------+    "Search and retrieve info"          |            |
|  | EXECUTE  |<-------------------------------+      |            |
|  +----+-----+                                |      |            |
|       v                                      |      |            |
|  +----------+    "Write an answer"           |      |            |
|  | GENERATE +--------------------------------+      |            |
|  +----+-----+                                |      |            |
|       v                                      |      |            |
|  +----------+    "Rate quality 0-10"         |      |            |
|  | CRITIQUE +--------------------------------+      |            |
|  +----+-----+                                      |            |
|       |                                            |            |
|       | if score < 7 AND retries < max             |            |
|       +-------------- RETRY LOOP ------------------+            |
|                                                                    |
|       | if score >= 7 OR retries >= max                           |
|       v                                                            |
|  +----------+                                                     |
|  |   END    |                                                     |
|  +----------+                                                     |
+-------------------------------------------------------------------+
"""
print(diagram)

# --- NODES ---

def plan_node(state):
    """Node 1: Create a plan based on the query."""
    query = state["query"]
    plan = f"Search for '{query}', then summarize findings"
    print(f"  [PLAN] {plan}")
    return {"plan": plan}

def execute_node(state):
    """Node 2: Execute the plan (simulate search)."""
    results = f"Found info about: {state['query']}. LangGraph is a graph-based agent framework."
    print(f"  [EXECUTE] Got results")
    return {"search_results": results}

def generate_node(state):
    """Node 3: Generate an answer from search results."""
    answer = f"Based on research: {state['search_results']}. LangGraph gives branching and self-correction."
    print(f"  [GENERATE] Generated answer")
    return {"answer": answer}

def critique_node(state):
    """Node 4: Critique the answer and assign quality score."""
    retry = state["retry_count"]
    score = 5 if retry == 0 else 8
    print(f"  [CRITIQUE] Score: {score}/10 (attempt #{retry + 1})")
    return {
        "quality_score": score,
        "retry_count": retry + 1,
    }

# --- CONDITIONAL ROUTER ---
def route_after_critique(state):
    score = state["quality_score"]
    retries = state["retry_count"]
    max_retries = state["max_retries"]
    if score >= 7:
        print(f"  [ROUTE] Score {score} >= 7 -> FINISH")
        return "finish"
    elif retries < max_retries:
        print(f"  [ROUTE] Score {score} < 7, retries {retries} < {max_retries} -> RETRY")
        return "retry"
    else:
        print(f"  [ROUTE] Max retries reached -> FINISH")
        return "finish"

# --- BUILD THE GRAPH ---
graph = StateGraph(ResearchState)

graph.add_node("plan", plan_node)
graph.add_node("execute", execute_node)
graph.add_node("generate", generate_node)
graph.add_node("critique", critique_node)

graph.add_edge("plan", "execute")
graph.add_edge("execute", "generate")
graph.add_edge("generate", "critique")

graph.add_conditional_edges("critique", route_after_critique, {
    "retry": "execute",
    "finish": END
})

graph.set_entry_point("plan")
app = graph.compile()

# --- RUN IT ---
print("=" * 60)
result = app.invoke({
    "query": "What is LangGraph?",
    "plan": "", "search_results": "", "answer": "",
    "quality_score": 0, "retry_count": 0, "max_retries": 3,
})

print(f"\nFINAL ANSWER: {result['answer'][:80]}...")
print(f"QUALITY SCORE: {result['quality_score']}/10")
print(f"RETRIES: {result['retry_count']}")


+-------------------------------------------------------------------+
|                    RESEARCH ASSISTANT GRAPH                        |
+-------------------------------------------------------------------+
|  +----------+                                                     |
|  |  START   |                                                     |
|  +----+-----+                                                     |
|       v                                                            |
|  +----------+    "Create a plan based on the query"               |
|  |   PLAN   +----------------------------------------+            |
|  +----+-----+                                        |            |
|       v                                              |            |
|  +----------+    "Search and retrieve info"          |            |
|  | EXECUTE  |<-------------------------------+      |            |
|  +----+-----+                                |      |            |
|       v          

In [ ]:
# =============================================================================
# TASK 2: Build a Linear Graph
# =============================================================================
# 4 nodes in a straight line: PLAN -> EXECUTE -> GENERATE -> FORMAT
# Uses Day 2 tools + LLM (Groq)
# =============================================================================

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from typing import TypedDict
from langgraph.graph import StateGraph, END

load_dotenv()

# --- LLM ---
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, max_tokens=1024)

# --- TOOLS (from Day 2) ---
@tool
def get_weather(city: str) -> str:
    """Returns current temperature for a given city. Use when user asks about weather."""
    fake_temps = {"tokyo": 22, "paris": 18, "new york": 25, "london": 15}
    temp = fake_temps.get(city.lower(), 20)
    return f"Weather in {city.title()}: {temp}C"

@tool
def product_lookup(product_name: str) -> str:
    """Looks up product info including price and stock. Use when user asks about products."""
    products = {"laptop": 999, "phone": 699, "tablet": 449, "headphones": 149}
    price = products.get(product_name.lower())
    if price:
        return f"Product: {product_name}, Price: ${price}"
    return f"Product '{product_name}' not found"

tools = [get_weather, product_lookup]

# --- STATE SCHEMA ---
class ResearchState(TypedDict):
    query: str
    plan: str
    tool_output: str
    answer: str
    formatted_output: str

# --- NODE 1: PLAN ---
def plan_node(state):
    """Use LLM to create a plan for answering the query."""
    response = llm.invoke([
        {"role": "system", "content": "You are a planner. Create a short step-by-step plan. Reply with just the plan."},
        {"role": "user", "content": state["query"]}
    ])
    plan = response.content
    print(f"  [PLAN] {plan[:120]}...")
    return {"plan": plan}

# --- NODE 2: EXECUTE ---
def execute_node(state):
    """Execute the plan by calling appropriate tools."""
    query = state["query"].lower()

    if "weather" in query or "temperature" in query:
        city = "paris"
        for c in ["tokyo", "paris", "new york", "london"]:
            if c in query:
                city = c
                break
        result = get_weather.invoke({"city": city})
    elif "product" in query or "price" in query or "laptop" in query or "phone" in query:
        product = "laptop"
        for p in ["laptop", "phone", "tablet", "headphones"]:
            if p in query:
                product = p
                break
        result = product_lookup.invoke({"product_name": product})
    else:
        response = llm.invoke([
            {"role": "system", "content": "Answer briefly in 1-2 sentences."},
            {"role": "user", "content": state["query"]}
        ])
        result = response.content

    print(f"  [EXECUTE] {result[:120]}")
    return {"tool_output": result}

# --- NODE 3: GENERATE ---
def generate_node(state):
    """Use LLM to generate a full answer from the tool output."""
    response = llm.invoke([
        {"role": "system", "content": "You are a helpful assistant. Given the query and tool results, write a clear answer."},
        {"role": "user", "content": f"Query: {state['query']}\nTool output: {state['tool_output']}"}
    ])
    answer = response.content
    print(f"  [GENERATE] {answer[:120]}...")
    return {"answer": answer}

# --- NODE 4: FORMAT ---
def format_node(state):
    """Format the final answer for display."""
    formatted = f"=== FINAL ANSWER ===\n{state['answer']}"
    print(f"  [FORMAT] Done")
    return {"formatted_output": formatted}

# --- BUILD THE GRAPH ---
graph = StateGraph(ResearchState)

graph.add_node("plan", plan_node)
graph.add_node("execute", execute_node)
graph.add_node("generate", generate_node)
graph.add_node("format", format_node)

graph.add_edge("plan", "execute")
graph.add_edge("execute", "generate")
graph.add_edge("generate", "format")
graph.add_edge("format", END)

graph.set_entry_point("plan")
app = graph.compile()

# --- RUN IT ---
print("=" * 60)
print("RUNNING LINEAR GRAPH")
print("=" * 60)
result = app.invoke({
    "query": "What's the weather in Paris?",
    "plan": "",
    "tool_output": "",
    "answer": "",
    "formatted_output": "",
})

print()
print(result["formatted_output"])
print()
print("STATE AFTER GRAPH:")
print(f"  Plan: {result['plan'][:80]}...")
print(f"  Tool Output: {result['tool_output']}")
print(f"  Answer: {result['answer'][:80]}...")